In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import sent_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
import pickle



In [2]:
FILE_PATH = "/kaggle/input/datasets/ronikdedhia/next-word-prediction/1661-0.txt"

with open(FILE_PATH, "r", encoding="utf-8") as file:
    raw_text = file.read()

print("Characters:", len(raw_text))
print(raw_text[:500])

Characters: 581888
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Charac


In [3]:
print(raw_text[:500])

﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Charac


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)             
    text = re.sub(r"http\S+|www\S+", " ", text)   
    text = re.sub(r"\d+", " ", text)               
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()       
    return text




In [5]:
processed_text = clean_text(raw_text)


In [6]:
print("Words:", len(processed_text.split()))
print(processed_text[:500])

Words: 107490
﻿ project gutenbergs the adventures of sherlock holmes by arthur conan doyle this ebook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever you may copy it give it away or reuse it under the terms of the project gutenberg license included with this ebook or online at title the adventures of sherlock holmes author arthur conan doyle release date november ebook last updated may language english character set encoding utf start of this project gutenberg ebook the


In [7]:
MAX_VOCABULARY = 5000

tokenizer = Tokenizer(
    num_words=MAX_VOCABULARY,
    oov_token="<OOV>"
)


In [8]:
tokenizer.fit_on_texts([processed_text])


In [11]:
vocab_size = min(
    MAX_VOCABULARY,
    len(tokenizer.word_index) + 1
)

print("Vocabulary size:", vocab_size)

token_sequence = tokenizer.texts_to_sequences([processed_text])[0]

token_sequence = [
    token for token in token_sequence
    if 1 < token < vocab_size
]

print("Total tokens:", len(token_sequence))

Vocabulary size: 5000
Total tokens: 102188


In [14]:
SEQUENCE_LENGTH = 9

In [15]:
input_sequences = []

for i in range(SEQUENCE_LENGTH, len(token_sequence)):
    sequence = token_sequence[i - SEQUENCE_LENGTH:i]
    input_sequences.append(sequence)

input_sequences = np.array(input_sequences)



In [16]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (102179, 8)
y shape: (102179,)


In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))

Training examples: 81743
Validation examples: 20436


In [18]:
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH - 1,)),

    Embedding(input_dim=vocab_size, output_dim=128),

    Bidirectional(LSTM(256, dropout=0.2, recurrent_dropout=0.1)),

    Dropout(0.3),

    Dense(256, activation="relu"),
    Dropout(0.2),

    Dense(vocab_size, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1787862185.666742      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787862185.673135      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 8, 128)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 512)            │       788,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5000)           │     1,285,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,844,808 (10.85 MB)

 Trainable params: 2,844,808 (10.85 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)



In [20]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping]
)

Epoch 1/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 37s 45ms/step - accuracy: 0.0593 - loss: 6.3576 - val_accuracy: 0.0721 - val_loss: 6.0763
Epoch 2/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.0815 - loss: 5.9068 - val_accuracy: 0.0955 - val_loss: 5.8349
Epoch 3/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 43ms/step - accuracy: 0.1121 - loss: 5.5821 - val_accuracy: 0.1200 - val_loss: 5.6447
Epoch 4/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.1289 - loss: 5.3407 - val_accuracy: 0.1278 - val_loss: 5.5770
Epoch 5/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.1398 - loss: 5.1445 - val_accuracy: 0.1324 - val_loss: 5.5427
Epoch 6/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 43ms/step - accuracy: 0.1510 - loss: 4.9626 - val_accuracy: 0.1376 - val_loss: 5.5615
Epoch 7/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 43ms/step - accuracy: 0.1601 - loss: 4.7866 - val_accuracy: 0.1370 - val_loss: 5.5833
Epoch 8/30
639/639 ━━━━━━━━━━━━━━━━━━━━ 28s 43ms/step - accuracy: 0.1698 - loss: 4.6262 - 

In [21]:
test_loss, test_accuracy = model.evaluate(
    X_val,
    y_val,
    verbose=1
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

639/639 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.1324 - loss: 5.5427
Test Loss: 5.542722225189209
Test Accuracy: 0.13241338729858398


In [22]:
model.save("next_word_predictor.keras")

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("Model and tokenizer saved successfully.")

Model and tokenizer saved successfully.


In [23]:
def predict_next_word(seed_text, model, tokenizer, sequence_length=9, temperature=0.7):
    cleaned_seed = clean_text(seed_text)
    sequence = tokenizer.texts_to_sequences([cleaned_seed])[0]
    sequence = sequence[-(sequence_length - 1):]

    padded = pad_sequences(
        [sequence],
        maxlen=sequence_length - 1,
        padding="pre"
    )

    probabilities = model.predict(padded, verbose=0)[0]

    probabilities = np.log(probabilities + 1e-10) / temperature
    probabilities = np.exp(probabilities)
    probabilities = probabilities / np.sum(probabilities)

    predicted_token = np.random.choice(
        len(probabilities),
        p=probabilities
    )

    return tokenizer.index_word.get(predicted_token, "")

In [24]:
seed_text = "the king was"
next_word = predict_next_word(seed_text, model, tokenizer)

print(f"Input: {seed_text}")
print(f"Predicted next word: {next_word}")

Input: the king was
Predicted next word: not


In [25]:
def generate_text(seed_text, num_words, model, tokenizer):
    generated_text = seed_text

    for _ in range(num_words):
        next_word = predict_next_word(generated_text, model, tokenizer)
        generated_text += " " + next_word

    return generated_text

print(generate_text(
    "it was a dark",
    num_words=10,
    model=model,
    tokenizer=tokenizer
))

it was a dark of a few thing as he told you that he
